# 09 — InceptionV3 pipeline on a Colab GPU runtime (attached from VSCode)

Use this when you've connected VSCode to a **Colab GPU runtime**. Important: the kernel and filesystem live on Colab's remote VM, **not** your Mac — so your local repo files aren't visible here. This notebook therefore clones the repo and rebuilds the dataset *on the VM*, then runs the whole InceptionV3 pipeline on `--device cuda`.

Difference from `08_colab_inception_pipeline.ipynb` (which is meant for the Colab web UI): this one **retrains the model on the GPU** by default, avoiding the Google-Drive auth popup that's unreliable over a VSCode-attached runtime. Drive is offered as an optional alternative if you want to reuse your exact checkpoint.

Run cells top-to-bottom. See [`docs/inception_v3_experiment.md`](https://github.com/1minute99/DSC291_Research_Reproduction/blob/inception-extension/docs/inception_v3_experiment.md) for what each stage means.

In [1]:
# 0. Confirm the remote kernel actually has a GPU.
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- attach a GPU runtime!')

CUDA available: True
device: NVIDIA A100-SXM4-80GB


In [2]:
# 1. Clone the repo + MILAN submodule onto the Colab VM.
REPO_URL = 'https://github.com/1minute99/DSC291_Research_Reproduction.git'
BRANCH   = 'inception-extension'
REPO     = '/content/DSC291_Research_Reproduction'

import os
if not os.path.isdir(REPO):
    !git clone --recurse-submodules --branch {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git submodule update --init --recursive
!git log --oneline -1

Cloning into '/content/DSC291_Research_Reproduction'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 95 (delta 32), reused 83 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 34.76 MiB | 41.05 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Submodule 'milan' (https://github.com/evandez/neuron-descriptions) registered for path 'milan'
Cloning into '/content/DSC291_Research_Reproduction/milan'...
remote: Enumerating objects: 4861, done.        
remote: Counting objects: 100% (223/223), done.        
remote: Compressing objects: 100% (66/66), done.        
remote: Total 4861 (delta 159), reused 199 (delta 149), pack-reused 4638 (from 1)        
Receiving objects: 100% (4861/4861), 3.13 MiB | 31.70 MiB/s, done.
Resolving deltas: 100% (3336/3336), done.
Submodule path 'milan': checked out '19c4d58f849d82be8ad23cd3bd47908a0eacd8ed'
/content/DSC291_Research_Reproduction

In [3]:
# 2. Install deps. KEEP Colab's preinstalled CUDA torch/torchvision (don't let
#    milan/requirements.in's CPU pins override them); SKIP allennlp (the repo's
#    _shims/allennlp handles it via milan_glue/upstream.py).
!grep -ivE '^[[:space:]]*(allennlp|torch|torchvision)([[:space:]<>=!]|$)' milan/requirements.in > /tmp/req_milan_colab.txt
!pip install -q -r /tmp/req_milan_colab.txt
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 118.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 89.0 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 28.1 MB/s et

In [4]:
# 3. Environment variables + import paths (mirror environment.md).
import os, sys
os.environ['MILAN_DATA_DIR']    = f'{REPO}/data'
os.environ['MILAN_MODELS_DIR']  = f'{REPO}/models'
os.environ['MILAN_RESULTS_DIR'] = f'{REPO}/results'
os.environ['PYTHONPATH']        = f'{REPO}:{REPO}/milan'   # for `!python -m ...`
for p in (REPO, f'{REPO}/milan'):
    if p not in sys.path:
        sys.path.insert(0, p)
print('env set')

env set


In [7]:
# 4. Rebuild the spurious dataset (deterministic, seed=0 -> identical to local).
#    Downloads Imagenette (~325 MB) on first run.
!python -m milan_repro.data.build_splits

downloading Imagenette to /content/DSC291_Research_Reproduction/data/imagenette/imagenette2-320.tgz ...
extracting /content/DSC291_Research_Reproduction/data/imagenette/imagenette2-320.tgz ...
/content/DSC291_Research_Reproduction/milan_repro/data/build_splits.py:83: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall(target_dir)
wrote 13394 images, manifest at /content/DSC291_Research_Reproduction/data/imagenet-spurious-text/50pct/manifest.csv


In [8]:
# 5. Train InceptionV3 on the GPU (default path -- self-contained, no Drive).
#    ~minutes on a T4/A100. Writes models/inception_v3_spurious.pth.
!python -m milan_repro.train.train_inception_v3 --device cuda

epoch   1  train_loss=3.6632  val_loss=1.9475  val_acc=0.4784
epoch   2  train_loss=2.8164  val_loss=1.2434  val_acc=0.6072
epoch   3  train_loss=2.0631  val_loss=1.1679  val_acc=0.7012
epoch   4  train_loss=1.4910  val_loss=0.7672  val_acc=0.7497
epoch   5  train_loss=1.1478  val_loss=0.6435  val_acc=0.7909
epoch   6  train_loss=0.9706  val_loss=0.6052  val_acc=0.7983
epoch   7  train_loss=0.8376  val_loss=0.6119  val_acc=0.8120
epoch   8  train_loss=0.7379  val_loss=0.5900  val_acc=0.8268
epoch   9  train_loss=0.6482  val_loss=0.5946  val_acc=0.8300
epoch  10  train_loss=0.5865  val_loss=0.5595  val_acc=0.8247
epoch  11  train_loss=0.5044  val_loss=0.5755  val_acc=0.8342
epoch  12  train_loss=0.4589  val_loss=0.6319  val_acc=0.8226
epoch  13  train_loss=0.4150  val_loss=0.6154  val_acc=0.8342
epoch  14  train_loss=0.4021  val_loss=0.6362  val_acc=0.8363
epoch  15  train_loss=0.3447  val_loss=0.6128  val_acc=0.8268
epoch  16  train_loss=0.3048  val_loss=0.6511  val_acc=0.8353
epoch  1

> **Optional — reuse your exact local checkpoint instead of retraining.** If Drive auth works over your VSCode connection, upload `inception_v3_spurious.pth` to `MyDrive/dsc291/` and run:
> ```python
> from google.colab import drive; drive.mount('/content/drive')
> !cp /content/drive/MyDrive/dsc291/inception_v3_spurious.pth models/
> ```
> Then skip cell 5.

In [9]:
# 6. Exemplars (dissection): top-k activating regions for all ~3,936 units at 299px.
!python -m milan_repro.milan_glue.run_exemplars --arch inception_v3 --device cuda

[Conv2d_2b_3x3] dissecting...
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 30 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
tally activations:   0% 0/74 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 30 worker processes in total. Our suggested max number of worker in current system is 12, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  se

In [10]:
# 7. Descriptions: MILAN `base` decoder captions each unit (downloads ~1 GB
#    decoder + ResNet101 encoder weights on first run).
!python -m milan_repro.milan_glue.run_descriptions \
    --dissect-dir results/edit/imagenet-spurious-text/inception_v3_spurious-50pct \
    --out results/descriptions_inception_v3.csv --device cuda

load imagenet-spurious-text/inception_v3_spurious-50pct: 100% 5/5 [00:14<00:00,  2.90s/it]
100% 324M/324M [00:08<00:00, 41.1MB/s] 
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth
100% 171M/171M [00:00<00:00, 199MB/s] 
decoding descriptions for 3936 units...
predict captions: 100% 246/246

In [11]:
# 8. Flag text neurons (description contains text/word/letter).
!python -m milan_repro.editing.identify_text_neurons \
    --descriptions results/descriptions_inception_v3.csv \
    --out results/descriptions_inception_v3_annotated.csv

2182/3936 units flagged as text neurons; wrote results/descriptions_inception_v3_annotated.csv


In [12]:
# 9. Editing experiment: per-unit importance + text-sorted / sort-all / random
#    curves. The slow stage on MPS; far faster on CUDA. Raise --ablation-step /
#    lower --ablation-max to trim further.
!python -m milan_repro.editing.evaluate --arch inception_v3 --device cuda.,

load imagenet-spurious-text/inception_v3_spurious-50pct: 100% 5/5 [00:14<00:00,  2.96s/it]
2182 text-neuron candidates
per-unit importance: 100% 3936/3936 [3:43:18<00:00,  3.40s/it]  
baseline: clean(val)=0.8247  adv(test)=0.1720
wrote ablation curve to /content/DSC291_Research_Reproduction/results/ablation_curve_inception_v3.csv


In [13]:
# 10. Figures + recovery summary.
!python -m milan_repro.figures.plot_fig7 \
    --dissect-dir results/edit/imagenet-spurious-text/inception_v3_spurious-50pct \
    --descriptions results/descriptions_inception_v3_annotated.csv \
    --out results/figs/fig7_inception_v3.pdf
!python -m milan_repro.figures.plot_fig8 \
    --ablation-csv results/ablation_curve_inception_v3.csv \
    --out results/figs/fig8_inception_v3.pdf \
    --title 'InceptionV3 robustness vs. ablation (MILAN Section 7, extension)'

import pandas as pd
df = pd.read_csv('results/ablation_curve_inception_v3.csv')
base = df[df['mode']=='baseline'].iloc[0]['adv_acc']
best = df[df['mode'].isin(['text-sorted','sort-all'])].groupby('mode')['adv_acc'].max()
print('adv baseline:', round(base,4))
print('text-sorted recovery: +{:.1f} pt'.format(100*(best.get('text-sorted',float('nan'))-base)))
print('sort-all   recovery: +{:.1f} pt'.format(100*(best.get('sort-all',float('nan'))-base)))

load imagenet-spurious-text/inception_v3_spurious-50pct: 100% 5/5 [00:14<00:00,  2.92s/it]
wrote results/figs/fig7_inception_v3.pdf
wrote results/figs/fig8_inception_v3.pdf
adv baseline: 0.172
text-sorted recovery: +4.8 pt
sort-all   recovery: +6.5 pt


In [14]:
# 11. Bundle the small result artifacts so you can pull them back to your Mac.
#     In VSCode: right-click the .zip in the remote Explorer -> Download,
#     or copy it to Drive if mounted.
!cd results && zip -q -r /content/inception_results.zip \
    descriptions_inception_v3.csv descriptions_inception_v3_annotated.csv \
    ablation_curve_inception_v3.csv importance_inception_v3.csv figs 2>/dev/null
!ls -lh /content/inception_results.zip

-rw-r--r-- 1 root root 4.6M May 31 00:07 /content/inception_results.zip


In [17]:
from google.colab import drive
drive.mount('/content/drive')                          # approve the auth prompt
!mkdir -p /content/drive/MyDrive/dsc291
!cp /content/inception_results.zip /content/drive/MyDrive/dsc291/
!ls -lh /content/drive/MyDrive/dsc291/inception_results.zip


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
-rw------- 1 root root 4.6M May 31 00:11 /content/drive/MyDrive/dsc291/inception_results.zip


In [16]:
from google.colab import drive; drive.mount('/content/drive')
DEST = '/content/drive/MyDrive/dsc291/results_inception'
!mkdir -p {DEST}
!cp results/descriptions_inception_v3*.csv \
    results/ablation_curve_inception_v3.csv \
    results/importance_inception_v3.csv {DEST}/ 2>/dev/null
!cp -r results/figs {DEST}/
!ls -lh {DEST} {DEST}/figs


Mounted at /content/drive
/content/drive/MyDrive/dsc291/results_inception:
total 367K
-rw------- 1 root root  18K May 31 00:08 ablation_curve_inception_v3.csv
-rw------- 1 root root 136K May 31 00:08 descriptions_inception_v3_annotated.csv
-rw------- 1 root root 119K May 31 00:08 descriptions_inception_v3.csv
drwx------ 2 root root 4.0K May 31 00:08 figs
-rw------- 1 root root  91K May 31 00:08 importance_inception_v3.csv

/content/drive/MyDrive/dsc291/results_inception/figs:
total 4.5M
-rw------- 1 root root 4.5M May 31 00:08 fig7_inception_v3.pdf
-rw------- 1 root root  19K May 31 00:08 fig8_inception_v3.pdf
